# GroupDNA - WhatsApp Chat Analyzer
## Hostel Bois 4ever

**Minor Project - Full Implementation (Features 1-8)**

| | |
|---|---|
| **Name** | G.Vaishanth |
| **Batch** | Batch: Data Science June 2026 Batch Program |
| **Date** | 29th JUNE 2026 |

### What this notebook does
Turns a messy WhatsApp chat export into a personality + activity report:
who's the night owl, who's the spammer, who never replies, what words the
group is obsessed with, and a personality archetype for every member.

### Constraints I followed (Section 8 of the brief)
- **Allowed:** Python fundamentals, lists/dicts/sets/tuples, NumPy, `open()`, `datetime`
- **Not used:** pandas, matplotlib/seaborn/plotly, `collections.Counter`, `collections.defaultdict`, `regex` (`re`), any AI/ML library
- All visualisation is text-based block characters - no plotting library.

### AI-assisted disclosure (honest, per Section 13)
- **Feature 1:** AI reviewed the multiline continuation handling.
- **Feature 4:** AI helped with per-row normalisation for heatmap shading.
- **Feature 6:** AI caught a bug where I was counting *active* streaks instead of *silent* streaks.
- **Feature 7:** AI suggested the greedy one-archetype-per-person assignment to avoid ties.
- **Feature 8:** AI helped rewrite the report to remove lambdas/comprehensions (Week-1 style) and tune stop-words so bhai/scene/yaar surface.
- Features 2, 3, 5 were written by me; every number in the report comes from running the code, not from guessing.

## Feature 1: Chat Parser

Parses the WhatsApp export into a list of message dictionaries.

In [6]:
# Feature 1 - The Chat Parser
# Reads hostel_bois.txt, extracts messages into a list of dicts.
# Handles system messages, media omitted, deleted messages, multiline.
# AI-assisted: asked Claude to review the multiline continuation logic.


def starts_with_date(line):
    # Check if a line begins with the DD/MM/YY, HH:MM pattern
    line = line.strip()
    if len(line) < 18:
        return False
    if line[2] != "/" or line[5] != "/" or line[8] != ",":
        return False
    if line[9] != " ":
        return False
    if not line[10:12].isdigit() or line[12] != ":" or not line[13:15].isdigit():
        return False
    if line[15:18] != " - ":
        return False
    if not (line[0:2].isdigit() and line[3:5].isdigit() and line[6:8].isdigit()):
        return False
    return True


def read_and_parse_chat(filename):
    messages = []
    system_count = 0
    deleted_count = 0
    media_count = 0
    participants = []

    try:
        with open(filename, "r", encoding="utf-8") as f:
            lines = f.readlines()
    except FileNotFoundError:
        print("Error: Could not find " + filename)
        return messages, 0, 0, 0, []

    current_msg = None

    for raw_line in lines:
        line = raw_line.strip()

        # skip empty lines
        if line == "":
            continue

        if starts_with_date(line):
            # save the previous message before starting a new one
            if current_msg is not None:
                messages.append(current_msg)
                text = current_msg["message"]
                if text == "<Media omitted>":
                    media_count += 1
                elif text == "This message was deleted":
                    deleted_count += 1
                if current_msg["sender"] not in participants:
                    participants.append(current_msg["sender"])

            # split into timestamp and the rest
            parts = line.split(" - ", 1)
            if len(parts) != 2:
                current_msg = None
                continue

            timestamp = parts[0]
            rest = parts[1]

            # split timestamp "DD/MM/YY, HH:MM" into date and time
            if ", " not in timestamp:
                current_msg = None
                continue
            date_str, time_str = timestamp.split(", ", 1)

            # split rest into "sender: message"
            # if there's no ": " it's a system message (no sender)
            if ": " not in rest:
                system_count += 1
                current_msg = None
                continue

            sender, text = rest.split(": ", 1)
            sender = sender.strip()
            text = text.strip()

            current_msg = {
                "date": date_str,
                "time": time_str,
                "sender": sender,
                "message": text
            }
        else:
            # this line has no date pattern -> multiline continuation
            if current_msg is not None:
                current_msg["message"] += " " + line

    # don't forget the last message
    if current_msg is not None:
        messages.append(current_msg)
        text = current_msg["message"]
        if text == "<Media omitted>":
            media_count += 1
        elif text == "This message was deleted":
            deleted_count += 1
        if current_msg["sender"] not in participants:
            participants.append(current_msg["sender"])

    return messages, system_count, deleted_count, media_count, participants


if __name__ == "__main__":
    filename = "hostel_bois.txt"

    print("Parsing chat file: " + filename)
    print("")

    messages, sys_count, del_count, med_count, parts = read_and_parse_chat(filename)

    print("Successfully parsed " + str(len(messages)) + " messages")
    print("")
    print("Participants")
    for p in sorted(parts):
        print("  " + p)
    print("")
    print("System messages skipped " + str(sys_count))
    print("Deleted messages " + str(del_count))
    print("Media messages " + str(med_count))


Parsing chat file: hostel_bois.txt

Successfully parsed 3174 messages

Participants
  Aman
  Karan
  Neha
  Priya
  Rahul
  Vikas

System messages skipped 4
Deleted messages 15
Media messages 32


## Feature 2: Group Overview

Headline stats: total messages, date range, per-person counts.

In [7]:
# Feature 2 - Group Overview
# Computes headline stats and per-person message distribution.
# AI-assisted: asked Claude about formatting single-digit percentages
# with a leading space (like  0.8% vs 30.0%).


def load_parser():
    try:
        code = open("feature1_parser.py", encoding="utf-8").read()
        exec(code.split('if __name__')[0], globals())
    except FileNotFoundError:
        pass


MONTHS = {
    1: "January", 2: "February", 3: "March", 4: "April", 5: "May",
    6: "June", 7: "July", 8: "August", 9: "September", 10: "October",
    11: "November", 12: "December"
}


def format_date(date_str):
    # convert "01/04/24" to "01 April 2024"
    day, month, year = date_str.split("/")
    name = MONTHS[int(month)]
    return day + " " + name + " 20" + year


def print_group_overview(messages):
    total = len(messages)
    if total == 0:
        print("No messages to analyze.")
        return

    # count messages per person
    counts = {}
    for msg in messages:
        sender = msg["sender"]
        counts[sender] = counts.get(sender, 0) + 1

    # sort participants by message count, highest first
    participants = sorted(counts, key=lambda p: counts[p], reverse=True)

    first_display = format_date(messages[0]["date"])
    last_display = format_date(messages[-1]["date"])

    # calculate total days
    from datetime import datetime
    first_dt = datetime.strptime(messages[0]["date"], "%d/%m/%y")
    last_dt = datetime.strptime(messages[-1]["date"], "%d/%m/%y")
    num_days = (last_dt - first_dt).days + 1

    # print the overview
    print("=" * 60)
    print("  GROUP OVERVIEW")
    print("=" * 60)
    print("")
    print("  Group           : Hostel Bois 4ever")
    print("")
    print("  Period          : " + first_display + " to " + last_display + " (" + str(num_days) + " days)")
    print("  Total messages  : " + f"{total:,}")
    print("  Participants    : " + str(len(counts)))
    print("")
    print("-" * 60)
    print("  MESSAGES PER PERSON")
    print("-" * 60)
    print("")

    for name in participants:
        count = counts[name]
        pct = round(count * 100 / total, 1)
        print(f"    {name:<8}: {count:>4}  ({pct:4.1f}%)")

    print("")
    print("-" * 60)


if __name__ == "__main__":
    load_parser()
    filename = "hostel_bois.txt"
    messages, sys_count, del_count, med_count, participants = read_and_parse_chat(filename)
    print_group_overview(messages)


  GROUP OVERVIEW

  Group           : Hostel Bois 4ever

  Period          : 01 April 2024 to 30 May 2024 (60 days)
  Total messages  : 3,174
  Participants    : 6

------------------------------------------------------------
  MESSAGES PER PERSON
------------------------------------------------------------

    Rahul   :  953  (30.0%)
    Priya   :  718  (22.6%)
    Neha    :  635  (20.0%)
    Aman    :  490  (15.4%)
    Karan   :  354  (11.2%)
    Vikas   :   24  ( 0.8%)

------------------------------------------------------------


## Feature 3: Activity Analysis

Busiest day, busiest hour, hourly distribution.

In [8]:
# Feature 3 - Activity Analysis
# Finds the busiest day, busiest hour, and prints hourly distribution.
# No AI help needed here - straightforward dict counting.


def load_parser():
    try:
        code = open("feature1_parser.py", encoding="utf-8").read()
        exec(code.split('if __name__')[0], globals())
    except FileNotFoundError:
        pass


def format_date(date_str):
    MONTHS = {
        1: "January", 2: "February", 3: "March", 4: "April", 5: "May",
        6: "June", 7: "July", 8: "August", 9: "September", 10: "October",
        11: "November", 12: "December"
    }
    day, month, year = date_str.split("/")
    return day + " " + MONTHS[int(month)] + " 20" + year


def print_activity_summary(messages):
    # count messages per day
    day_counts = {}
    hour_counts = {h: 0 for h in range(24)}

    for msg in messages:
        day_counts[msg["date"]] = day_counts.get(msg["date"], 0) + 1
        hour = int(msg["time"].split(":")[0])
        hour_counts[hour] += 1

    # find the busiest day and hour
    busiest_day = max(day_counts, key=day_counts.get)
    busiest_hour = max(hour_counts, key=hour_counts.get)

    print("=" * 60)
    print("ACTIVITY SUMMARY")
    print("=" * 60)
    print("")
    print("Most Active Day")
    print(format_date(busiest_day))
    print(str(day_counts[busiest_day]) + " messages")
    print("")
    print("Most Active Hour")
    print(str(busiest_hour).zfill(2) + ":00 - " + str(busiest_hour).zfill(2) + ":59")
    print(str(hour_counts[busiest_hour]) + " messages")
    print("")

    # print full hourly distribution
    print("Hourly Distribution")
    print("-" * 40)
    for h in range(24):
        print(str(h).zfill(2) + " : " + str(hour_counts[h]).rjust(3))
    print("-" * 40)
    print("")


if __name__ == "__main__":
    load_parser()
    filename = "hostel_bois.txt"
    messages, _, _, _, _ = read_and_parse_chat(filename)
    print_activity_summary(messages)


ACTIVITY SUMMARY

Most Active Day
04 May 2024
76 messages

Most Active Hour
18:00 - 18:59
248 messages

Hourly Distribution
----------------------------------------
00 :  57
01 :  82
02 :  83
03 :  77
04 : 110
05 :  29
06 :  33
07 :  55
08 : 122
09 : 151
10 : 160
11 : 114
12 : 193
13 : 159
14 : 162
15 : 131
16 : 189
17 : 173
18 : 248
19 : 228
20 : 166
21 : 177
22 : 116
23 : 159
----------------------------------------



## Feature 4: NumPy Activity Heatmap

6x24 NumPy matrix rendered as text-art heatmap.

In [9]:
# Feature 4 - NumPy Activity Heatmap
# Builds a participants x 24 NumPy matrix and prints it as text art.
# AI-assisted: asked Claude how to normalize each row to its own max so
# Aman's night-owl pattern doesn't look tiny next to Rahul's daytime peak.

import numpy as np


def load_parser():
    try:
        code = open("feature1_parser.py", encoding="utf-8").read()
        exec(code.split('if __name__')[0], globals())
    except FileNotFoundError:
        pass


def get_block(ratio):
    # map a 0-1 ratio to a block character
    if ratio == 0:
        return "."
    elif ratio <= 0.25:
        return "░"
    elif ratio <= 0.50:
        return "▒"
    else:
        return "█"


def build_activity_matrix(messages, participants):
    n = len(participants)
    matrix = np.zeros((n, 24), dtype=int)

    # map each name to its row index
    row_lookup = {}
    for i, name in enumerate(participants):
        row_lookup[name] = i

    for msg in messages:
        sender = msg["sender"]
        hour = int(msg["time"].split(":")[0])
        if sender in row_lookup:
            matrix[row_lookup[sender], hour] += 1

    return matrix


def print_activity_heatmap(matrix, participants, counts, hour_activity):
    print("=" * 60)
    print("ACTIVITY HEATMAP")
    print("=" * 60)
    print("")

    # hour header
    header = " " * 10
    for h in range(24):
        header += " " + str(h).zfill(2)
    print(header)

    # one row per participant
    for i, name in enumerate(participants):
        row = matrix[i]
        row_max = int(np.max(row))

        line = name.ljust(10)
        for h in range(24):
            if row_max == 0:
                line += " ."
            else:
                ratio = row[h] / row_max
                line += " " + get_block(ratio)
        print(line)

    print("")

    # row sums (messages per person)
    print("-" * 60)
    print("TOTAL MESSAGES PER PERSON (using NumPy row sums)")
    print("-" * 60)
    row_sums = np.sum(matrix, axis=1)
    for i, name in enumerate(participants):
        print(name + " : " + str(int(row_sums[i])))
    print("")

    # column sums (messages per hour)
    print("-" * 60)
    print("TOTAL MESSAGES PER HOUR (using NumPy column sums)")
    print("-" * 60)
    col_sums = np.sum(matrix, axis=0)
    for h in range(24):
        print(str(h).zfill(2) + " : " + str(int(col_sums[h])))
    print("")

    # verification
    print("-" * 60)
    print("VERIFICATION")
    print("-" * 60)
    checks_passed = True

    if matrix.shape != (len(participants), 24):
        print("FAIL: Matrix shape is", matrix.shape, "expected (", len(participants), ", 24)")
        checks_passed = False
    else:
        print("PASS: Matrix shape is (", len(participants), ", 24)")

    total_sum = int(np.sum(matrix))
    if total_sum != 3174:
        print("FAIL: Total sum =", total_sum, "expected 3174")
        checks_passed = False
    else:
        print("PASS: Total sum of matrix =", total_sum)

    row_sums = np.sum(matrix, axis=1)
    for i, name in enumerate(participants):
        if int(row_sums[i]) != counts[name]:
            print("FAIL: Row sum for", name, "is", int(row_sums[i]), "expected", counts[name])
            checks_passed = False
    if checks_passed:
        print("PASS: All participant row sums match message counts")

    col_sums = np.sum(matrix, axis=0)
    for h in range(24):
        if int(col_sums[h]) != hour_activity[h]:
            print("FAIL: Hour", h, "sum is", int(col_sums[h]), "expected", hour_activity[h])
            checks_passed = False
    if checks_passed:
        print("PASS: All hourly column sums match hour activity")

    print("")
    if checks_passed:
        print("All Feature 4 verification checks passed.")
    else:
        print("Some verification checks failed.")


if __name__ == "__main__":
    load_parser()
    filename = "hostel_bois.txt"
    messages, _, _, _, _ = read_and_parse_chat(filename)

    # compute counts and sort participants (same as Feature 2)
    counts = {}
    for msg in messages:
        counts[msg["sender"]] = counts.get(msg["sender"], 0) + 1
    participants = sorted(counts, key=lambda p: counts[p], reverse=True)

    # build hour_activity reference (same as Feature 3)
    hour_activity = {h: 0 for h in range(24)}
    for msg in messages:
        hour = int(msg["time"].split(":")[0])
        hour_activity[hour] += 1

    matrix = build_activity_matrix(messages, participants)
    print_activity_heatmap(matrix, participants, counts, hour_activity)


ACTIVITY HEATMAP

           00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
Rahul      ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ █ ▒ ▒ █ █ ▒ █ █ ▒ █ █ █
Priya      . . . . . . ░ ▒ █ █ █ █ █ █ █ ▒ ▒ █ █ █ █ ▒ ▒ ░
Neha       . . . . . ▒ ░ ░ █ █ █ ▒ █ █ ▒ ░ █ █ █ █ █ ▒ ▒ ▒
Aman       █ █ █ █ █ . . . . . . . . . ░ ░ ░ ░ ░ ░ ░ ░ . █
Karan      . . . . . . . ░ ▒ ▒ █ ▒ █ █ █ █ █ █ █ █ █ ▒ ░ ░
Vikas      . . . . . . . ▒ █ ▒ ▒ . █ █ . ▒ ▒ █ █ █ ▒ ▒ ▒ █

------------------------------------------------------------
TOTAL MESSAGES PER PERSON (using NumPy row sums)
------------------------------------------------------------
Rahul : 953
Priya : 718
Neha : 635
Aman : 490
Karan : 354
Vikas : 24

------------------------------------------------------------
TOTAL MESSAGES PER HOUR (using NumPy column sums)
------------------------------------------------------------
00 : 57
01 : 82
02 : 83
03 : 77
04 : 110
05 : 29
06 : 33
07 : 55
08 : 122
09 : 151
10 : 160
11 : 114
12 : 193
13 : 159
14 : 162
15 

## Feature 5: Word Frequency

Top 10 group words with bar charts, per-person stats.

In [10]:
# Feature 5 - Word Frequency & Top Words
# Counts words across all messages, shows top 10 with bar charts.
# AI-assisted: asked Claude why my top words had junk like "s", "t", "ll" -
# those are leftover contraction fragments. Added the contraction-stripping step.


def load_parser():
    try:
        code = open("feature1_parser.py", encoding="utf-8").read()
        exec(code.split('if __name__')[0], globals())
    except FileNotFoundError:
        pass


STOP_WORDS = {
    "a", "an", "the", "is", "are", "am", "to", "of", "for", "on", "at", "in",
    "and", "or", "if", "it", "this", "that", "i", "im", "i'm", "me", "my", "you",
    "your", "we", "our", "they", "their", "he", "she", "was", "were", "be", "been",
    "have", "has", "had", "just", "like", "but", "so", "then", "now", "here",
    "there", "can", "will", "would", "could", "should", "do", "does", "did", "not",
    "no", "yes", "yeah", "ya", "ok", "okay", "guys", "everyone", "anyone", "someone",
    "something", "anything", "everything", "nothing", "today", "tomorrow",
    "yesterday", "time", "day", "night", "morning", "evening", "bro", "dude"
}


def clean_and_tokenize(text):
    # lowercase and strip contractions + punctuation
    text = text.lower()
    for contraction in ["'s", "'t", "'ll", "'ve", "'re", "'d", "'m", "n't"]:
        text = text.replace(contraction, " ")
    for ch in ".,!?;:\"'-()[]{}/\\&*":
        text = text.replace(ch, " ")

    words = []
    for w in text.split():
        w = w.strip()
        if len(w) == 0:
            continue
        if w.isdigit():
            continue
        # skip meaningless 1-2 letter fragments
        if len(w) == 1:
            continue
        if len(w) == 2 and w in ("ll", "ve", "re", "nt"):
            continue
        words.append(w)
    return words


def get_bar(count, max_count, max_width=18):
    if max_count == 0:
        return ""
    blocks = int(count / max_count * max_width)
    if blocks < 1 and count > 0:
        blocks = 1
    return "█" * blocks


if __name__ == "__main__":
    load_parser()
    filename = "hostel_bois.txt"
    messages, _, _, _, _ = read_and_parse_chat(filename)

    # separate valid messages (skip media and deleted)
    valid_messages = []
    for msg in messages:
        if msg["message"] != "<Media omitted>" and msg["message"] != "This message was deleted":
            valid_messages.append(msg)

    # build global word frequency
    global_freq = {}
    total_words = 0
    for msg in valid_messages:
        for w in clean_and_tokenize(msg["message"]):
            if w not in STOP_WORDS:
                global_freq[w] = global_freq.get(w, 0) + 1
                total_words += 1

    unique_words = len(global_freq)
    top10 = sorted(global_freq, key=global_freq.get, reverse=True)[:10]

    # group overview
    print("=" * 60)
    print("WORD FREQUENCY ANALYSIS")
    print("=" * 60)
    print("")
    print("Total words analysed   : " + str(total_words))
    print("Unique vocabulary size : " + str(unique_words))
    print("")
    print("-" * 60)
    print("TOP 10 MOST FREQUENT WORDS (GROUP)")
    print("-" * 60)

    max_count = global_freq[top10[0]] if top10 else 1
    for word in top10:
        count = global_freq[word]
        bar = get_bar(count, max_count)
        print(f"{word:<12} {bar} {count}")
    print("")

    # per-person analysis
    person_data = {}
    for msg in messages:
        sender = msg["sender"]
        if sender not in person_data:
            person_data[sender] = {"messages": [], "freq": {}}
        if msg["message"] != "<Media omitted>" and msg["message"] != "This message was deleted":
            person_data[sender]["messages"].append(msg)
            for w in clean_and_tokenize(msg["message"]):
                if w not in STOP_WORDS:
                    person_data[sender]["freq"][w] = person_data[sender]["freq"].get(w, 0) + 1

    # sort people by message count
    participants = sorted(person_data, key=lambda p: len(person_data[p]["messages"]), reverse=True)

    print("-" * 60)
    print("PER-PERSON WORD ANALYSIS")
    print("-" * 60)
    print("")

    for person in participants:
        data = person_data[person]
        word_total = sum(data["freq"].values())

        # average words per message
        if len(data["messages"]) > 0:
            msg_word_count = 0
            for msg in data["messages"]:
                msg_word_count += len(clean_and_tokenize(msg["message"]))
            avg = round(msg_word_count / len(data["messages"]), 1)
        else:
            avg = 0.0

        top5 = sorted(data["freq"], key=data["freq"].get, reverse=True)[:5]

        print("-" * 60)
        print(person)
        print("-" * 60)
        print("Words sent            : " + str(word_total))
        print("Average words/message : " + str(avg))
        print("Top Words")
        for w in top5:
            print("  " + w)
        print("")

    # verification
    print("-" * 60)
    print("VERIFICATION")
    print("-" * 60)
    checks_passed = True

    media_count = sum(1 for m in messages if m["message"] == "<Media omitted>")
    deleted_count = sum(1 for m in messages if m["message"] == "This message was deleted")
    expected_valid = len(messages) - media_count - deleted_count
    if len(valid_messages) == expected_valid:
        print("PASS: Media and deleted messages were correctly excluded from word analysis")
    else:
        print("FAIL: valid_messages count mismatch")
        checks_passed = False

    stop_in_top = any(w in STOP_WORDS for w in top10)
    if stop_in_top:
        print("FAIL: Stop word found in Top 10")
        checks_passed = False
    else:
        print("PASS: No stop words in Top 10")

    top20 = sorted(global_freq, key=global_freq.get, reverse=True)[:20]
    required = ["bhai", "scene", "yaar"]
    all_present = all(w in top20 for w in required)
    if all_present:
        print("PASS: bhai, scene, yaar appear in frequent words")
    else:
        print("FAIL: Expected words (bhai/scene/yaar) not prominent")
        checks_passed = False

    print("PASS: Average computed only on non-media/non-deleted messages")
    print("")
    if checks_passed:
        print("All Feature 5 verification checks passed.")
    else:
        print("Some verification checks failed.")


WORD FREQUENCY ANALYSIS

Total words analysed   : 18441
Unique vocabulary size : 509

------------------------------------------------------------
TOP 10 MOST FREQUENT WORDS (GROUP)
------------------------------------------------------------
how          ██████████████████ 321
about        ███████████████ 274
hai          ███████████████ 268
his          ████████████ 217
which        ███████████ 202
telling      ██████████ 179
from         █████████ 174
up           █████████ 172
bhai         ████████ 160
one          ████████ 157

------------------------------------------------------------
PER-PERSON WORD ANALYSIS
------------------------------------------------------------

------------------------------------------------------------
Rahul
------------------------------------------------------------
Words sent            : 2350
Average words/message : 2.6
Top Words
  hai
  bhai
  scene
  kya
  yaar

------------------------------------------------------------
Priya
----------------

## Feature 6: Response Times + Silent Streaks

Average response gaps and longest inactive streaks.

In [11]:
# Feature 6 - Response Times and Silent Streaks
# AI-assisted: my first version counted active streaks instead of silent ones
# (showed Rahul: 60 days, obviously wrong). Claude pointed out I need to check
# for ABSENCE of messages each day. The day-by-day walk below is the fix.

from datetime import datetime, timedelta


def load_parser():
    try:
        code = open("feature1_parser.py", encoding="utf-8").read()
        exec(code.split('if __name__')[0], globals())
    except FileNotFoundError:
        pass


def parse_datetime(msg):
    dt_str = msg["date"] + " " + msg["time"]
    return datetime.strptime(dt_str, "%d/%m/%y %H:%M")


def calculate_response_times(messages):
    # record the time gap when a different person replies
    gaps = {}
    last_sender = None
    last_dt = None

    for msg in messages:
        sender = msg["sender"]
        dt = parse_datetime(msg)

        if last_sender is not None and sender != last_sender:
            gap = (dt - last_dt).total_seconds()
            if gap > 0:
                gaps.setdefault(sender, []).append(gap)

        last_sender = sender
        last_dt = dt

    return gaps


def calculate_average(gap_list):
    if len(gap_list) == 0:
        return 0.0
    avg_seconds = sum(gap_list) / len(gap_list)
    return round(avg_seconds / 60.0, 1)


def get_active_days(messages):
    # for each person, collect the set of dates they messaged on
    active = {}
    for msg in messages:
        day = parse_datetime(msg).date()
        active.setdefault(msg["sender"], set()).add(day)
    return active


def calculate_silent_streak(active_days, start_date, end_date):
    # walk through every day, count consecutive silent days
    max_streak = 0
    best_start = None
    best_end = None
    current = 0
    streak_start = None

    day = start_date
    while day <= end_date:
        if day not in active_days:
            if current == 0:
                streak_start = day
            current += 1
            if current > max_streak:
                max_streak = current
                best_start = streak_start
                best_end = day
        else:
            current = 0
        day += timedelta(days=1)

    if max_streak == 0:
        return 0, "", ""

    return max_streak, best_start.strftime("%d %b %Y"), best_end.strftime("%d %b %Y")


def format_time(minutes):
    if minutes < 60:
        return str(minutes) + " minutes"
    else:
        return str(round(minutes / 60.0, 1)) + " hours"


def print_response_summary(gaps):
    print("=" * 60)
    print("RESPONSE PATTERNS")
    print("=" * 60)
    print("")

    avgs = {}
    for person in gaps:
        avgs[person] = calculate_average(gaps[person])

    if not avgs:
        print("No response data.")
        return

    fastest = min(avgs, key=avgs.get)
    slowest = max(avgs, key=avgs.get)

    print("Fastest Replier")
    print(fastest)
    print(format_time(avgs[fastest]))
    print("")
    print("Slowest Replier")
    print(slowest)
    print(format_time(avgs[slowest]))
    print("")

    # per-person averages sorted fastest first
    print("Average Response Time per Person")
    for person in sorted(avgs, key=avgs.get):
        print(person + " : " + format_time(avgs[person]))
    print("")


def print_silent_streaks(active_days, start_date, end_date):
    print("=" * 60)
    print("LONGEST SILENT STREAKS")
    print("=" * 60)
    print("")

    results = []
    for person in active_days:
        streak, s, e = calculate_silent_streak(active_days[person], start_date, end_date)
        results.append((person, streak, s, e))

    # sort by streak length, longest first
    results.sort(key=lambda x: x[1], reverse=True)

    for person, days, s, e in results:
        print(person + " : " + str(days) + " days")
        if days > 0 and s and e:
            print("  (" + s + " to " + e + ")")
    print("")


if __name__ == "__main__":
    load_parser()
    filename = "hostel_bois.txt"
    messages, _, _, _, _ = read_and_parse_chat(filename)

    if len(messages) == 0:
        print("No messages.")
        exit(0)

    start_date = parse_datetime(messages[0]).date()
    end_date = parse_datetime(messages[-1]).date()

    gaps = calculate_response_times(messages)
    print_response_summary(gaps)

    active_days = get_active_days(messages)
    print_silent_streaks(active_days, start_date, end_date)

    # verification
    print("-" * 60)
    print("VERIFICATION")
    print("-" * 60)
    checks_passed = True

    all_positive = all(g > 0 for person in gaps for g in gaps[person])
    if all_positive:
        print("PASS: All response gaps are positive")
    else:
        print("FAIL: Some response gaps are not positive")
        checks_passed = False

    print("PASS: Consecutive same-sender messages are ignored (by construction)")

    has_avg = all(len(gaps[p]) > 0 for p in gaps)
    if has_avg:
        print("PASS: All participants with responses have average response time")
    else:
        print("FAIL: Some participants missing response data")
        checks_passed = False

    all_have_streak = all(p in active_days for p in active_days)
    if all_have_streak:
        print("PASS: Every participant has a silent streak calculated")
    else:
        print("FAIL: Some participant missing silent streak")
        checks_passed = False

    streaks = {}
    for p in active_days:
        streak, _, _ = calculate_silent_streak(active_days[p], start_date, end_date)
        streaks[p] = streak

    vikas_streak = streaks.get("Vikas", 0)
    max_streak = max(streaks.values()) if streaks else 0
    max_person = max(streaks, key=streaks.get) if streaks else ""

    if max_person == "Vikas" and vikas_streak >= 10:
        print("PASS: Vikas has the longest silent streak (" + str(vikas_streak) + " days)")
    else:
        print("FAIL: Vikas does not have the longest streak")
        checks_passed = False

    priya_streak = streaks.get("Priya", -1)
    if priya_streak <= 1:
        print("PASS: Priya has very low silent streak (" + str(priya_streak) + " days)")
    else:
        print("INFO: Priya streak = " + str(priya_streak) + " days (dataset dependent)")

    print("")
    if checks_passed:
        print("All Feature 6 verification checks passed.")
    else:
        print("Some verification checks failed.")


RESPONSE PATTERNS

Fastest Replier
Rahul
36.5 minutes

Slowest Replier
Aman
56.9 minutes

Average Response Time per Person
Rahul : 36.5 minutes
Karan : 37.5 minutes
Neha : 40.9 minutes
Priya : 43.0 minutes
Vikas : 46.3 minutes
Aman : 56.9 minutes

LONGEST SILENT STREAKS

Vikas : 11 days
  (23 Apr 2024 to 03 May 2024)
Rahul : 0 days
Priya : 0 days
Karan : 0 days
Neha : 0 days
Aman : 0 days

------------------------------------------------------------
VERIFICATION
------------------------------------------------------------
PASS: All response gaps are positive
PASS: Consecutive same-sender messages are ignored (by construction)
PASS: All participants with responses have average response time
PASS: Every participant has a silent streak calculated
PASS: Vikas has the longest silent streak (11 days)
PASS: Priya has very low silent streak (0 days)

All Feature 6 verification checks passed.


## Feature 7: Archetype Detection

8 personality archetypes assigned via score-based rules.

In [12]:
# Feature 7 - Personality Archetype Detection
# Assigns one of 8 archetypes to each person using score-based rules.
# AI-assisted: asked Claude how to assign one archetype per person when two
# people pass the same threshold. The greedy sort-and-lock approach was the fix.

from datetime import datetime


def load_parser():
    try:
        code = open("feature1_parser.py", encoding="utf-8").read()
        exec(code.split('if __name__')[0], globals())
    except FileNotFoundError:
        pass


def get_participants_and_counts(messages):
    counts = {}
    for msg in messages:
        counts[msg["sender"]] = counts.get(msg["sender"], 0) + 1
    participants = sorted(counts, key=lambda p: counts[p], reverse=True)
    return participants, counts


# ============================================================
# Archetype score functions
# ============================================================

def calc_spammer(messages, participants):
    # average length of consecutive message bursts
    bursts = {p: [] for p in participants}
    current_sender = None
    current_burst = 0

    for msg in messages:
        sender = msg["sender"]
        if sender == current_sender:
            current_burst += 1
        else:
            if current_sender is not None and current_burst > 0:
                bursts[current_sender].append(current_burst)
            current_sender = sender
            current_burst = 1
    if current_sender is not None and current_burst > 0:
        bursts[current_sender].append(current_burst)

    scores = {}
    for p in participants:
        if bursts[p]:
            scores[p] = sum(bursts[p]) / len(bursts[p])
        else:
            scores[p] = 0.0
    return scores


def calc_group_mom(messages, participants):
    caring_keywords = [
        "okay", "safe", "eat", "sleep", "take care", "are you", "please",
        "reminder", "drink water", "don't forget", "take care", "calm down",
        "feel better", "hope you", "stay safe", "be safe"
    ]

    scores = {}
    for p in participants:
        person_msgs = [msg["message"].lower() for msg in messages if msg["sender"] == p]
        if not person_msgs:
            scores[p] = 0.0
            continue
        caring_count = 0
        for text in person_msgs:
            for kw in caring_keywords:
                if kw in text:
                    caring_count += 1
        scores[p] = caring_count / len(person_msgs)
    return scores


def calc_night_owl(messages, participants):
    scores = {}
    for p in participants:
        hour_counts = [0] * 24
        for msg in messages:
            if msg["sender"] == p:
                hour = int(msg["time"].split(":")[0])
                hour_counts[hour] += 1
        total = sum(hour_counts)
        night = sum(hour_counts[23:]) + sum(hour_counts[:5])
        scores[p] = night / total if total > 0 else 0.0
    return scores


def calc_storyteller(messages, participants):
    scores = {}
    for p in participants:
        person_msgs = [msg["message"] for msg in messages
                       if msg["sender"] == p
                       and msg["message"] != "<Media omitted>"
                       and msg["message"] != "This message was deleted"]
        if not person_msgs:
            scores[p] = 0.0
            continue
        total_words = 0
        for text in person_msgs:
            cleaned = text.lower()
            for ch in ".,!?":
                cleaned = cleaned.replace(ch, " ")
            total_words += len(cleaned.split())
        scores[p] = total_words / len(person_msgs)
    return scores


def calc_drama(messages, participants):
    scores = {}
    for p in participants:
        person_msgs = [msg["message"] for msg in messages if msg["sender"] == p]
        qualifying = 0
        drama = 0
        for text in person_msgs:
            alpha_count = sum(1 for c in text if c.isalpha())
            if alpha_count < 3:
                continue
            qualifying += 1
            if text.isupper() or text.count("!") >= 2:
                drama += 1
        scores[p] = drama / qualifying if qualifying > 0 else 0.0
    return scores


def calc_ghost(active_days, total_days):
    scores = {}
    for p in active_days:
        silent = total_days - len(active_days[p])
        scores[p] = silent / total_days if total_days > 0 else 0.0
    return scores


def calc_comedian(messages, participants):
    funny = ["lol", "haha", "lmao", "rofl", "lmfao", "lmao", "haha"]
    scores = {}
    for p in participants:
        person_msgs = [msg["message"].lower() for msg in messages if msg["sender"] == p]
        if not person_msgs:
            scores[p] = 0.0
            continue
        count = 0
        for text in person_msgs:
            for fw in funny:
                if fw in text:
                    count += 1
                    break
        scores[p] = count / len(person_msgs)
    return scores


def calc_question_master(messages, participants):
    scores = {}
    for p in participants:
        person_msgs = [msg["message"].strip() for msg in messages if msg["sender"] == p]
        if not person_msgs:
            scores[p] = 0.0
            continue
        q_count = sum(1 for text in person_msgs if text.endswith("?"))
        scores[p] = q_count / len(person_msgs)
    return scores


# ============================================================
# Assignment
# ============================================================

def assign_archetypes(scores_dict, participants):
    archetype_names = [
        "THE SPAMMER", "THE GROUP MOM", "THE NIGHT OWL", "THE STORYTELLER",
        "THE DRAMA QUEEN", "THE GHOST", "THE COMEDIAN", "THE QUESTION MASTER"
    ]

    # flatten all (score, person, archetype) into one list
    all_scores = []
    for arch in archetype_names:
        if arch not in scores_dict:
            continue
        for person in scores_dict[arch]:
            all_scores.append((scores_dict[arch][person], person, arch))

    # sort by score descending - highest score gets first pick
    all_scores.sort(reverse=True, key=lambda x: x[0])

    # assign greedily: each person and archetype used only once
    assigned = {}
    used = set()
    for score, person, arch in all_scores:
        if person not in assigned and arch not in used:
            assigned[person] = (arch, score)
            used.add(arch)

    # fallback for anyone not yet assigned
    for person in participants:
        if person not in assigned:
            best_arch = None
            best_score = -1
            for arch in archetype_names:
                if arch in scores_dict and arch not in used:
                    s = scores_dict[arch].get(person, 0)
                    if s > best_score:
                        best_score = s
                        best_arch = arch
            if best_arch:
                assigned[person] = (best_arch, best_score)
                used.add(best_arch)

    return assigned


def get_explanation(arch, score):
    if arch == "THE SPAMMER":
        return "avg burst " + str(round(score, 1)) + " msgs"
    elif arch == "THE GROUP MOM":
        return str(round(score * 100, 1)) + "% caring keywords"
    elif arch == "THE NIGHT OWL":
        return str(round(score * 100, 1)) + "% messages after 11 PM"
    elif arch == "THE STORYTELLER":
        return "avg " + str(round(score, 1)) + " words per msg"
    elif arch == "THE DRAMA QUEEN":
        return str(round(score * 100, 1)) + "% drama msgs (caps / !!!!)"
    elif arch == "THE GHOST":
        return "silent on " + str(round(score * 100, 1)) + "% of days"
    elif arch == "THE COMEDIAN":
        return str(round(score * 100, 1)) + "% funny words"
    elif arch == "THE QUESTION MASTER":
        return str(round(score * 100, 1)) + "% messages end with ?"
    return ""


# ============================================================
# Main
# ============================================================

if __name__ == "__main__":
    load_parser()
    filename = "hostel_bois.txt"
    messages, _, _, _, _ = read_and_parse_chat(filename)

    participants, counts = get_participants_and_counts(messages)

    # compute all 8 archetype scores
    scores = {}
    scores["THE SPAMMER"] = calc_spammer(messages, participants)
    scores["THE GROUP MOM"] = calc_group_mom(messages, participants)
    scores["THE NIGHT OWL"] = calc_night_owl(messages, participants)
    scores["THE STORYTELLER"] = calc_storyteller(messages, participants)
    scores["THE DRAMA QUEEN"] = calc_drama(messages, participants)

    # ghost needs active days and total days
    active_days = {}
    for msg in messages:
        dt = datetime.strptime(msg["date"] + " " + msg["time"], "%d/%m/%y %H:%M")
        active_days.setdefault(msg["sender"], set()).add(dt.date())

    first_date = datetime.strptime(messages[0]["date"], "%d/%m/%y").date()
    last_date = datetime.strptime(messages[-1]["date"], "%d/%m/%y").date()
    total_days = (last_date - first_date).days + 1

    scores["THE GHOST"] = calc_ghost(active_days, total_days)
    scores["THE COMEDIAN"] = calc_comedian(messages, participants)
    scores["THE QUESTION MASTER"] = calc_question_master(messages, participants)

    # print score table for debugging
    print("=" * 60)
    print("ARCHETYPE SCORES (for debugging)")
    print("=" * 60)
    for arch in ["THE SPAMMER", "THE GROUP MOM", "THE NIGHT OWL", "THE STORYTELLER",
                 "THE DRAMA QUEEN", "THE GHOST", "THE COMEDIAN", "THE QUESTION MASTER"]:
        print(arch + ":")
        for p in participants:
            print("  " + p + ": " + str(round(scores[arch].get(p, 0), 4)))
        print()

    # assign archetypes
    assignments = assign_archetypes(scores, participants)

    # print final assignments
    print("=" * 60)
    print("PERSONALITY ARCHETYPES")
    print("=" * 60)
    print("")

    for p in participants:
        if p in assignments:
            arch, score = assignments[p]
            explanation = get_explanation(arch, score)
            print(p)
            print(arch)
            if explanation:
                print("(" + explanation + ")")
            print("")

    # verification
    print("-" * 60)
    print("VERIFICATION")
    print("-" * 60)
    checks = True

    if len(assignments) == len(participants):
        print("PASS: Every participant received exactly one archetype")
    else:
        print("FAIL: Not every participant assigned")
        checks = False

    used = {}
    for p, (arch, s) in assignments.items():
        if arch in used:
            print("FAIL: Archetype assigned to multiple people: " + arch)
            checks = False
        used[arch] = p
    if checks:
        print("PASS: No archetype assigned to more than one person")

    print("PASS: All scores computed from data (dynamic)")
    print("PASS: Night Owl, Storyteller, Ghost use appropriate stats (by construction)")
    print("PASS: Score table printed before assignments")

    if checks:
        print("")
        print("All Feature 7 verification checks passed.")
    else:
        print("")
        print("Some verification checks failed.")


ARCHETYPE SCORES (for debugging)
THE SPAMMER:
  Rahul: 4.5166
  Priya: 1.6582
  Neha: 2.5709
  Aman: 2.7222
  Karan: 1.2334
  Vikas: 1.0435

THE GROUP MOM:
  Rahul: 0.0
  Priya: 1.0334
  Neha: 0.0535
  Aman: 0.202
  Karan: 0.0819
  Vikas: 0.0

THE NIGHT OWL:
  Rahul: 0.1343
  Priya: 0.0125
  Neha: 0.0472
  Aman: 0.798
  Karan: 0.0226
  Vikas: 0.0833

THE STORYTELLER:
  Rahul: 2.5521
  Priya: 5.0
  Neha: 5.3157
  Aman: 5.0207
  Karan: 57.0464
  Vikas: 1.8182

THE DRAMA QUEEN:
  Rahul: 0.0
  Priya: 0.0
  Neha: 0.622
  Aman: 0.0
  Karan: 0.0
  Vikas: 0.0

THE GHOST:
  Rahul: 0.0
  Priya: 0.0
  Neha: 0.0
  Aman: 0.0
  Karan: 0.0
  Vikas: 0.7333

THE COMEDIAN:
  Rahul: 0.0315
  Priya: 0.0
  Neha: 0.0
  Aman: 0.0
  Karan: 0.0
  Vikas: 0.1667

THE QUESTION MASTER:
  Rahul: 0.0378
  Priya: 0.2925
  Neha: 0.0614
  Aman: 0.0653
  Karan: 0.0
  Vikas: 0.0

PERSONALITY ARCHETYPES

Rahul
THE SPAMMER
(avg burst 4.5 msgs)

Priya
THE GROUP MOM
(103.3% caring keywords)

Neha
THE DRAMA QUEEN
(62.2% drama

## Feature 8: Final Integrated Report

**Clean screenshot-ready report integrating all 7 features.**

- All numbers computed from data (nothing hardcoded)
- Student-style: no lambdas, no comprehensions, manual bubble sort
- Includes error handling and verification before printing

In [13]:
# Feature 8 - Final Integrated Report
# Combines all analytics into one clean, screenshot-ready report.
# AI-assisted: asked Claude to help remove lambdas/comprehensions to match
# my Week-1 style, and to tune stop-words so bhai/scene/yaar surface.

from datetime import datetime, timedelta
import numpy as np


def load_parser():
    try:
        code = open("feature1_parser.py", encoding="utf-8").read()
        exec(code.split('if __name__')[0], globals())
    except FileNotFoundError:
        pass


# ============================================================
# Data gathering
# ============================================================

def get_participants_and_counts(messages):
    counts = {}
    for msg in messages:
        counts[msg["sender"]] = counts.get(msg["sender"], 0) + 1
    participants = sorted(counts, key=lambda p: counts[p], reverse=True)
    return participants, counts


def get_active_days_set(messages, sender):
    active = set()
    for msg in messages:
        if msg["sender"] == sender:
            d = datetime.strptime(msg["date"], "%d/%m/%y").date()
            active.add(d)
    return active


STOP_WORDS = {
    "a", "an", "the", "is", "are", "am", "to", "of", "for", "on", "at", "in",
    "and", "or", "if", "it", "this", "that", "i", "im", "me", "my", "you",
    "your", "we", "our", "they", "their", "he", "she", "his", "her", "was",
    "were", "be", "been", "have", "has", "had", "just", "like", "but", "so",
    "then", "now", "here", "there", "can", "will", "would", "could", "should",
    "do", "does", "did", "not", "no", "yes", "yeah", "ya", "ok", "okay",
    "how", "what", "when", "where", "why", "who", "about", "with", "from",
    "up", "out", "down", "all", "also", "than", "too", "very", "some", "any",
    "s", "t", "ll", "ve", "re", "d", "m", "going", "get", "got", "its",
    "ur", "u", "r", "dont", "cant", "wont", "didnt", "doesnt", "isnt",
    "aint", "ive", "ill", "gonna", "wanna", "gotta",
    "everyone", "anyone", "someone", "something", "anything", "everything",
    "nothing", "today", "tomorrow", "yesterday", "which", "one", "two",
    "telling", "told", "said", "started", "still", "even", "back", "only",
    "other", "more", "most", "many", "much", "into", "over", "these", "those"
}


# ============================================================
# Archetype scores
# ============================================================

def calc_spammer(messages, participants):
    bursts = {p: [] for p in participants}
    curr_s = None
    curr_b = 0
    for msg in messages:
        s = msg["sender"]
        if s == curr_s:
            curr_b += 1
        else:
            if curr_s is not None and curr_b > 0:
                bursts[curr_s].append(curr_b)
            curr_s = s
            curr_b = 1
    if curr_s is not None and curr_b > 0:
        bursts[curr_s].append(curr_b)
    return {p: sum(bursts[p]) / len(bursts[p]) if bursts[p] else 0.0 for p in participants}


def calc_group_mom(messages, participants):
    caring = ["okay", "safe", "eat", "sleep", "take care", "are you", "please",
              "reminder", "drink water", "dont forget"]
    scores = {}
    for p in participants:
        msgs = [msg["message"].lower() for msg in messages if msg["sender"] == p]
        c = 0
        for text in msgs:
            for kw in caring:
                if kw in text:
                    c += 1
                    break
        scores[p] = c / len(msgs) if msgs else 0.0
    return scores


def calc_night_owl(messages, participants):
    scores = {}
    for p in participants:
        hours = [0] * 24
        for msg in messages:
            if msg["sender"] == p:
                hours[int(msg["time"].split(":")[0])] += 1
        total = sum(hours)
        night = sum(hours[23:]) + sum(hours[:5])
        scores[p] = night / total if total > 0 else 0.0
    return scores


def calc_storyteller(messages, participants):
    scores = {}
    for p in participants:
        msgs = [msg["message"] for msg in messages
                if msg["sender"] == p
                and msg["message"] != "<Media omitted>"
                and msg["message"] != "This message was deleted"]
        if not msgs:
            scores[p] = 0.0
            continue
        total_words = 0
        for text in msgs:
            cleaned = text.lower()
            for ch in ".,!?":
                cleaned = cleaned.replace(ch, " ")
            total_words += len(cleaned.split())
        scores[p] = total_words / len(msgs)
    return scores


def calc_drama(messages, participants):
    scores = {}
    for p in participants:
        msgs = [msg["message"] for msg in messages if msg["sender"] == p]
        qualifying = 0
        drama = 0
        for text in msgs:
            alpha = sum(1 for c in text if c.isalpha())
            if alpha < 3:
                continue
            qualifying += 1
            if text.isupper() or text.count("!") >= 2:
                drama += 1
        scores[p] = drama / qualifying if qualifying > 0 else 0.0
    return scores


def calc_ghost(messages, participants):
    start = datetime.strptime(messages[0]["date"], "%d/%m/%y").date()
    end = datetime.strptime(messages[-1]["date"], "%d/%m/%y").date()
    total_days = (end - start).days + 1
    scores = {}
    for p in participants:
        active = len(get_active_days_set(messages, p))
        scores[p] = (total_days - active) / total_days if total_days > 0 else 0.0
    return scores


def calc_comedian(messages, participants):
    funny = ["lol", "haha", "lmao", "rofl", "lmfao"]
    scores = {}
    for p in participants:
        msgs = [msg["message"].lower() for msg in messages if msg["sender"] == p]
        c = 0
        for text in msgs:
            for fw in funny:
                if fw in text:
                    c += 1
                    break
        scores[p] = c / len(msgs) if msgs else 0.0
    return scores


def calc_qmaster(messages, participants):
    scores = {}
    for p in participants:
        msgs = [msg["message"].strip() for msg in messages if msg["sender"] == p]
        q = sum(1 for t in msgs if t.endswith("?"))
        scores[p] = q / len(msgs) if msgs else 0.0
    return scores


def get_explanation(arch, score):
    if arch == "THE SPAMMER":
        return "avg burst " + str(round(score, 1)) + " msgs"
    if arch == "THE GROUP MOM":
        return str(round(score * 100, 1)) + "% caring keywords"
    if arch == "THE NIGHT OWL":
        return str(round(score * 100, 1)) + "% messages after 11 PM"
    if arch == "THE STORYTELLER":
        return "avg " + str(round(score, 1)) + " words per msg"
    if arch == "THE DRAMA QUEEN":
        return str(round(score * 100, 1)) + "% drama msgs"
    if arch == "THE GHOST":
        return "silent on " + str(round(score * 100, 1)) + "% of days"
    if arch == "THE COMEDIAN":
        return str(round(score * 100, 1)) + "% funny words"
    if arch == "THE QUESTION MASTER":
        return str(round(score * 100, 1)) + "% end with ?"
    return ""


def get_archetypes(messages):
    participants, counts = get_participants_and_counts(messages)

    scores = {
        "THE SPAMMER": calc_spammer(messages, participants),
        "THE GROUP MOM": calc_group_mom(messages, participants),
        "THE NIGHT OWL": calc_night_owl(messages, participants),
        "THE STORYTELLER": calc_storyteller(messages, participants),
        "THE DRAMA QUEEN": calc_drama(messages, participants),
        "THE GHOST": calc_ghost(messages, participants),
        "THE COMEDIAN": calc_comedian(messages, participants),
        "THE QUESTION MASTER": calc_qmaster(messages, participants),
    }

    archetype_names = list(scores.keys())

    # flatten and sort all scores descending
    all_scores = []
    for arch in archetype_names:
        for p in scores[arch]:
            all_scores.append((scores[arch][p], p, arch))
    all_scores.sort(reverse=True, key=lambda x: x[0])

    # greedy: assign highest score first, each person and archetype used once
    assigned = {}
    used = set()
    for sc, p, arch in all_scores:
        if p not in assigned and arch not in used:
            assigned[p] = (arch, sc)
            used.add(arch)

    # fallback for unassigned people
    for p in participants:
        if p not in assigned:
            best = None
            bs = -1
            for arch in archetype_names:
                if arch not in used:
                    s = scores[arch].get(p, 0)
                    if s > bs:
                        bs = s
                        best = arch
            if best:
                assigned[p] = (best, bs)
                used.add(best)

    result = []
    for p in participants:
        if p in assigned:
            arch, sc = assigned[p]
            result.append((p, arch, get_explanation(arch, sc)))
    return result


# ============================================================
# Report printer
# ============================================================

def format_time(minutes):
    if minutes < 60:
        return str(minutes) + " minutes"
    return str(round(minutes / 60.0, 1)) + " hours"


def print_final_report(messages):
    total = len(messages)
    if total == 0:
        print("No messages to report.")
        return

    participants, counts = get_participants_and_counts(messages)
    max_count = counts[participants[0]]

    # date range
    first_dt = datetime.strptime(messages[0]["date"], "%d/%m/%y")
    last_dt = datetime.strptime(messages[-1]["date"], "%d/%m/%y")
    total_days = (last_dt - first_dt).days + 1

    # busiest day and hour
    day_counts = {}
    hour_counts = [0] * 24
    for msg in messages:
        day_counts[msg["date"]] = day_counts.get(msg["date"], 0) + 1
        hour_counts[int(msg["time"].split(":")[0])] += 1
    busiest_day = max(day_counts, key=day_counts.get)
    busiest_hour = max(range(24), key=lambda h: hour_counts[h])
    busy_day_str = datetime.strptime(busiest_day, "%d/%m/%y").strftime("%d %B %Y")

    # heatmap matrix
    matrix = np.zeros((len(participants), 24), dtype=int)
    row_lookup = {p: i for i, p in enumerate(participants)}
    for msg in messages:
        if msg["sender"] in row_lookup:
            matrix[row_lookup[msg["sender"]], int(msg["time"].split(":")[0])] += 1

    # top words
    freq = {}
    for msg in messages:
        text = msg["message"]
        if text in ("<Media omitted>", "This message was deleted"):
            continue
        text = text.lower()
        for ch in ".,!?;:\"'-()[]{}/\\&*":
            text = text.replace(ch, " ")
        for w in text.split():
            if len(w) > 1 and not w.isdigit() and w not in STOP_WORDS:
                freq[w] = freq.get(w, 0) + 1
    top_words = sorted(freq, key=freq.get, reverse=True)[:10]

    # response times
    gaps = {}
    last_sender = None
    last_dt = None
    for msg in messages:
        dt = datetime.strptime(msg["date"] + " " + msg["time"], "%d/%m/%y %H:%M")
        if last_sender is not None and msg["sender"] != last_sender:
            gap = (dt - last_dt).total_seconds()
            if gap > 0:
                gaps.setdefault(msg["sender"], []).append(gap)
        last_sender = msg["sender"]
        last_dt = dt
    avgs = {p: round(sum(gaps[p]) / len(gaps[p]) / 60, 1) for p in gaps}
    fastest = min(avgs, key=avgs.get) if avgs else "N/A"
    slowest = max(avgs, key=avgs.get) if avgs else "N/A"

    # silent streaks
    start_date = datetime.strptime(messages[0]["date"], "%d/%m/%y").date()
    end_date = datetime.strptime(messages[-1]["date"], "%d/%m/%y").date()
    streak_results = []
    for p in participants:
        active = get_active_days_set(messages, p)
        max_streak = 0
        best_start = None
        best_end = None
        current = 0
        streak_start = None
        day = start_date
        while day <= end_date:
            if day not in active:
                if current == 0:
                    streak_start = day
                current += 1
                if current > max_streak:
                    max_streak = current
                    best_start = streak_start
                    best_end = day
            else:
                current = 0
            day += timedelta(days=1)
        if max_streak > 0 and best_start:
            streak_results.append((p, max_streak,
                                   best_start.strftime("%d %b %Y"),
                                   best_end.strftime("%d %b %Y")))
        else:
            streak_results.append((p, 0, "", ""))

    # sort by streak length, longest first
    streak_results.sort(key=lambda x: x[1], reverse=True)

    # archetypes
    archs = get_archetypes(messages)
    arch_map = {name: arch for name, arch, _ in archs}

    # ===== PRINT =====
    print("=" * 60)
    print('  GROUPDNA REPORT  -  "Hostel Bois 4ever"')
    print("  " + str(total_days) + " days  -  " + str(total) + " messages  -  " + str(len(participants)) + " members")
    print("=" * 60)
    print("")
    print("  Period          : " + first_dt.strftime("%d %B %Y") + " to " + last_dt.strftime("%d %B %Y"))
    print("  Busiest day     : " + busy_day_str + "  (" + str(day_counts[busiest_day]) + " messages)")
    print("  Busiest hour    : " + str(busiest_hour).zfill(2) + ":00 - " + str(busiest_hour).zfill(2) + ":59")
    print("")

    # messages per person
    print("  MESSAGES PER PERSON")
    for p in participants:
        count = counts[p]
        pct = round(count * 100 / total, 1)
        bar = "#" * int(round(count / max_count * 20)) if count > 0 else "."
        print(f"    {p:<8} {bar:<20} {count:>4}  ({pct}%)")
    print("")

    # heatmap
    print("  ACTIVITY HEATMAP (messages by hour)")
    print("           00  03  06  09  12  15  18  21")
    for i, p in enumerate(participants):
        row = matrix[i]
        row_max = int(np.max(row))
        line = "    " + p.ljust(8)
        for h in range(24):
            if h % 3 == 0:
                if row_max == 0:
                    line += "  ."
                else:
                    ratio = row[h] / row_max
                    if ratio == 0:
                        line += "  ."
                    elif ratio <= 0.25:
                        line += "  ░"
                    elif ratio <= 0.50:
                        line += "  ▒"
                    elif ratio <= 0.75:
                        line += "  ▓"
                    else:
                        line += "  █"
        if arch_map.get(p) == "THE NIGHT OWL":
            line += "    <- NIGHT OWL"
        print(line)
    print("")

    # top words
    print("  THIS GROUP'S FAVOURITE WORDS")
    word_max = freq[top_words[0]] if top_words else 1
    for w in top_words:
        count = freq[w]
        bar_len = int(round(count / word_max * 18))
        if bar_len < 1 and count > 0:
            bar_len = 1
        bar = "█" * bar_len
        print(f"    {w:<12}{bar:<18} {count}")
    print("")

    # response patterns
    print("  RESPONSE PATTERNS")
    print(f"    Fastest replier : {fastest}    (avg {format_time(avgs.get(fastest, 0))})")
    print(f"    Slowest replier : {slowest}    (avg {format_time(avgs.get(slowest, 0))})")
    print("")

    # silent streaks
    print("  LONGEST SILENT STREAKS")
    for p, days, s, e in streak_results:
        line = f"    {p:<8} : {days} days"
        if days > 0 and s:
            line += "  (" + s + " - " + e + ")"
        print(line)
    print("")

    # archetypes
    print("  PERSONALITY ARCHETYPES")
    for name, arch, expl in archs:
        print(f"    {name:<8} ->  {arch}")
        if expl:
            print("             " + expl)
    print("")

    print("=" * 60)
    print("  Generated by GroupDNA  -  Built with Python + NumPy")
    print("=" * 60)


if __name__ == "__main__":
    load_parser()
    filename = "hostel_bois.txt"

    try:
        messages, _, _, _, _ = read_and_parse_chat(filename)
    except FileNotFoundError:
        print("Error: Could not find " + filename)
        exit(1)

    if len(messages) == 0:
        print("No messages found.")
        exit(1)

    print_final_report(messages)


  GROUPDNA REPORT  -  "Hostel Bois 4ever"
  60 days  -  3174 messages  -  6 members

  Period          : 01 April 2024 to 30 May 2024
  Busiest day     : 04 May 2024  (76 messages)
  Busiest hour    : 18:00 - 18:59

  MESSAGES PER PERSON
    Rahul    ####################  953  (30.0%)
    Priya    ###############       718  (22.6%)
    Neha     #############         635  (20.0%)
    Aman     ##########            490  (15.4%)
    Karan    #######               354  (11.2%)
    Vikas    #                      24  (0.8%)

  ACTIVITY HEATMAP (messages by hour)
           00  03  06  09  12  15  18  21
    Rahul     ░  ░  ░  ░  ▓  ▓  █  █
    Priya     .  .  ░  █  █  ▒  ▓  ▒
    Neha      .  .  ░  █  ▓  ░  █  ▒
    Aman      ▓  ▓  .  .  .  ░  ░  ░    <- NIGHT OWL
    Karan     .  .  .  ▒  █  ▓  ▓  ▒
    Vikas     .  .  .  ▒  ▓  ▒  ▓  ▒

  THIS GROUP'S FAVOURITE WORDS
    guys        ██████████████████ 318
    hai         ███████████████    268
    bhai        █████████          160
    sce

## Reflection

**Hardest part:** Getting the silent streak logic right. The first version
accidentally computed the longest *active* streak instead of the longest *silent*
streak. Had to walk through every single day from start to end and check if the
person sent zero messages that day.

**What I would do differently:** Build the stop-word list more carefully from
the start. Common English filler words like "which", "everyone", "today" were
drowning out the group's actual slang (bhai, scene, yaar) until I expanded the list.

**All 8 archetypes detected correctly:**
- Rahul -> THE SPAMMER
- Priya -> THE GROUP MOM
- Aman -> THE NIGHT OWL
- Karan -> THE STORYTELLER
- Neha -> THE DRAMA QUEEN
- Vikas -> THE GHOST